# GPU Assignment II — HW4.a + HW4.5 (ONE GPU)

**Single RTX 4090 / 24 GB.** Professor Savitha V's clarification (2026-10-27): one GPU is sufficient and Part C is optional.

**This is the SEED-corrected full re-run.** SEED is the instructor convention (last four digits of SJSU ID, **2738**), not the placeholder 42 used in the earlier run. Changing SEED changes the 70/15/15 split and the model initialisation, so **every training-based result is regenerated from scratch** here; the earlier seed-42 outputs are archived under `old_seed42_run/` and are *not* used in any graded table.

Run cells top to bottom from the repository root. Expected wall time on a 4090: ~1.5–2.5 h.

In [ ]:
from pathlib import Path
import os
import subprocess
import torch

# Find the project relative to the notebook's working directory
cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd / "gpu-assignment-ii",
    cwd.parent,
    cwd.parent / "gpu-assignment-ii",
]

ROOT = next(
    (p for p in candidates if (p / "configs" / "experiment.yaml").is_file()),
    None
)

assert ROOT is not None, f"Project not found. Current directory: {cwd}"

os.chdir(ROOT)

print("Project root:", ROOT)

subprocess.run(["nvidia-smi", "-L"], check=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA GPU required"

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

## 0 — Set SEED / SLICE, archive the old run, install missing dependencies

* `SEED = 2738` (SJSU ID convention, same as HW4.a).
* `SLICE` must equal the value used in HW4.a. Edit it below if HW4.a did not use 0.
* `bitsandbytes` was missing in the previous run (8-bit optimizer row was empty) — installed here.
* vLLM is installed in its own virtualenv `vllm_env/` so its pinned torch cannot clash with the training torch (the previous KV sweep crashed on `duplicate template name` from exactly that clash).

In [ ]:
import sys, shutil, yaml, datetime
from pathlib import Path

SEED  = 2738   # last four digits of SJSU ID — same as HW4.a
SLICE = 0      # <-- set to the SLICE used in HW4.a if it was not 0

cfg_path = ROOT / "configs" / "experiment.yaml"
cfg = yaml.safe_load(cfg_path.read_text())
cfg["seed"], cfg["slice"] = int(SEED), int(SLICE)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("experiment.yaml -> seed:", cfg["seed"], "slice:", cfg["slice"])

# Archive every previous result so seed-42 numbers can never mix with this run.
archive = ROOT / "old_seed42_run"
moved = 0
for d in ["results", "figures"]:
    src = ROOT / d
    if src.exists():
        for p in src.iterdir():
            if p.name == ".gitkeep":
                continue
            dest = archive / d / p.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
            shutil.move(str(p), str(dest)); moved += 1
for f in ["gpu_hours.csv"]:
    p = ROOT / f
    if p.exists() and p.stat().st_size > 60:
        dest = archive / f; dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dest)
# The dataset lock MUST be removed, otherwise common.py refuses the new seed.
for lock in [ROOT/"results/dataset_lock.json", archive/"results/dataset_lock.json"]:
    if lock.exists(): lock.unlink()
(ROOT/"results").mkdir(exist_ok=True); (ROOT/"figures").mkdir(exist_ok=True)
print(f"Archived {moved} items to {archive}")

with (ROOT/"RUN_LOG.txt").open("a") as f:
    f.write(f"\n{datetime.datetime.now(datetime.timezone.utc).isoformat()} | SEED corrected to {SEED} (SJSU ID convention), SLICE={SLICE}; previous seed-42 outputs archived to old_seed42_run/; full re-run started\n")

In [ ]:
%pip install -q -U pandas numpy pyyaml scikit-learn transformers datasets huggingface_hub accelerate matplotlib seaborn tqdm bitsandbytes

In [ ]:
# Verify bitsandbytes imports in THIS interpreter (8-bit optimizer experiment runs here).
import importlib, subprocess
r = subprocess.run([sys.executable, "-c", "import bitsandbytes, torch; print('bitsandbytes', bitsandbytes.__version__, '| torch', torch.__version__)"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "bitsandbytes failed to import — fix before Part B, otherwise the 8-bit row will be empty again"


In [ ]:
# vLLM in an isolated virtualenv so its pinned torch cannot break the training environment.
# VLLM_PY is used for every vLLM subprocess below. If vLLM already imports cleanly here, the current interpreter is used.
import subprocess, sys
from pathlib import Path

VENV = ROOT / "vllm_env"
VLLM_PY = str(VENV / "bin" / "python")

def vllm_ok(py):
    return subprocess.run([py, "-c", "import vllm, torch, yaml, pandas, sklearn, transformers; print('vllm', vllm.__version__)"],
                          capture_output=True, text=True).returncode == 0

if vllm_ok(sys.executable):
    VLLM_PY = sys.executable
    print("vLLM imports in the notebook interpreter; using it.")
else:
    if not Path(VLLM_PY).exists():
        subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
    subprocess.run([VLLM_PY, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
    # common.py needs pyyaml / pandas / scikit-learn inside the venv as well
    subprocess.run([VLLM_PY, "-m", "pip", "install", "-q", "vllm", "pyyaml", "pandas", "scikit-learn"], check=True)
    assert vllm_ok(VLLM_PY), "vLLM still fails to import inside vllm_env — check the pip output above"
    print("vLLM installed in", VENV)
print("VLLM_PY =", VLLM_PY)


## 1 — Dataset and reproducibility

`data/twitter.csv` **must be byte-identical to the file used in HW4.a.** If it already exists it is left untouched; it is only built from `tweet_eval` when missing. `SHUFFLE_SOURCE` must match what HW4.a did. The exact source row range is written to `results/row_range.json` and the dataset hash + SEED + SLICE are locked in `results/dataset_lock.json`.

In [ ]:
from datasets import load_dataset
import pandas as pd, hashlib

SHUFFLE_SOURCE = True   # True = same construction as the earlier run (tweet_eval shuffled once with random_state=42).
                        # Set False if HW4.a used the original tweet_eval ordering. Must match HW4.a.

data_dir = ROOT / "data"; data_dir.mkdir(parents=True, exist_ok=True)
output = data_dir / "twitter.csv"

if output.exists():
    print("Using existing", output, "(not rebuilt — must be the HW4.a file)")
else:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")
    df = pd.concat([ds[s].to_pandas() for s in ["train", "validation", "test"]], ignore_index=True)
    df = df[["text", "label"]].dropna()
    if SHUFFLE_SOURCE:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)   # source-file ordering only; NOT the split seed
    df.to_csv(output, index=False)
    print("Built:", output)

df = pd.read_csv(output)
print("Total rows:", len(df), "| classes:", df["label"].value_counts().to_dict())
print("SHA256:", hashlib.sha256(output.read_bytes()).hexdigest())

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from common import config, load_data

c = config()
assert int(c["seed"]) == SEED and int(c["slice"]) == SLICE, "experiment.yaml does not match the values set in section 0"

start = int(c["slice"]); subset = df.iloc[start:start + 5000]
print("SEED:", c["seed"], "| SLICE:", start, "| selected rows:", len(subset))
print("Subset class distribution:", subset["label"].value_counts().to_dict())
assert len(subset) > 0, "SLICE is outside the dataset"

tr, va, te, label_map = load_data(c)
print("\nTrain:", len(tr), "| Validation:", len(va), "| Test:", len(te), "| classes:", label_map)
print("\nrow_range.json:");   print((ROOT/"results/row_range.json").read_text())
print("dataset_lock.json:");  print((ROOT/"results/dataset_lock.json").read_text())

## 2 — HW4.a BERT fine-tuning
Train baseline and capture epoch-by-epoch loss, validation accuracy, peak GPU allocation, wall time, and best validation epoch.

In [ ]:
subprocess.run([sys.executable,'src/train_bert.py'],check=True)

## 3 — HW4.a test evaluation
Confusion matrix, per-class precision/recall, worst-class examples. **Write one-line diagnoses yourself** in the generated JSON; the script cannot determine qualitative causes.

In [ ]:
subprocess.run([sys.executable,'src/evaluate_bert.py'],check=True)
from IPython.display import display,Image
display(Image(filename='figures/training_curves.png'))
display(Image(filename='figures/confusion_matrix.png'))
print((ROOT/'results/worst_class_errors.json').read_text()[:5000])

## 4 — HW4.a serving baseline (Table HW4.1)

Separate causal LM (`serving_model` in the YAML). The vLLM runs use `VLLM_PY`; the HF pipeline baseline runs in the notebook interpreter. p50/p95 are **batch-completion latencies** attributed to each request in the batch — not TTFT or independently sampled per-request latency; say so in the write-up. This is a fresh `serving_metrics.csv`; the previous (seed-independent) rows are in `old_seed42_run/` for reference.

In [ ]:
import pandas as pd
for backend, b in [('hf', 1), ('vllm', 1), ('vllm', 8), ('vllm', 32)]:
    py = VLLM_PY if backend == 'vllm' else sys.executable
    log = ROOT / f"results/{backend}_batch_{b}.log"
    with log.open("w") as f:
        subprocess.run([py, 'src/benchmark_vllm.py', '--backend', backend, '--batch', str(b)], check=True, stdout=f, stderr=subprocess.STDOUT)
    print("done", backend, b, "->", log.name)
sm = pd.read_csv('results/serving_metrics.csv')
display(sm[[col for col in ['backend','batch','tokens_per_second','p50_batch_ms','p95_batch_ms','p50_request_ms','p95_request_ms','gpu_used_gb_from_nvidia_smi','status'] if col in sm.columns]])

## 5 — Part A: measured full-epoch batch scaling
Run successive batch sizes until OOM. Use measured samples/sec, peak allocated GPU memory and epoch seconds. Keep the same dataset, seed and training settings.


In [ ]:
subprocess.run([sys.executable,'src/full_experiments.py','--mode','scaling','--max-batch','1024'],check=True)
display(pd.read_csv('results/full_batch_scaling.csv'))


## 6 — Part B: independent techniques (incl. 8-bit optimizer) and data-selected combination

Baseline, AMP, gradient checkpointing, gradient accumulation and **8-bit AdamW (bitsandbytes, now installed)** each run independently with a batch scan to OOM. The script then selects a combination from the measured rows (`results/combination_selection.json`) and runs it. Accumulation is compared separately because it changes effective batch. Note from the previous run: the selector compares at the base batch (16), where AMP is overhead-bound; justify the final recommendation from the larger-batch rows in the write-up.

In [ ]:
subprocess.run([sys.executable,'src/full_experiments.py','--mode','techniques','--scan-technique-max','--max-batch','1024'],check=True)
mc = pd.read_csv('results/full_memory_comparison.csv')
display(mc)
bad = mc[mc.status.isin(['ERROR'])]
assert bad.empty, f"Some technique runs errored (not OOM):\n{bad[['technique','batch','details']]}"
assert (mc.technique=='8bit_optimizer').any() and (mc[mc.technique=='8bit_optimizer'].status=='ok').any(), "8-bit optimizer produced no successful row"


In [ ]:
# Equal-optimizer-step comparison for gradient accumulation (Part B.3): same number of steps, wall-clock and accuracy.
# Written to a separate CSV so the full-epoch table above is untouched.
import shutil
full = ROOT/'results/full_memory_comparison.csv'; tmp = ROOT/'results/_full_backup.csv'
shutil.copy2(full, tmp)
subprocess.run([sys.executable,'src/full_experiments.py','--mode','techniques','--fixed-steps','50'],check=True)
fixed = pd.read_csv(full); fixed = fixed[~fixed.index.isin(pd.read_csv(tmp).index)]
fixed.to_csv(ROOT/'results/fixed_steps_comparison.csv', index=False)
shutil.move(str(tmp), str(full))
display(fixed[['technique','batch','effective_batch','optimizer_steps','train_seconds','samples_per_second','peak_gb','eval_accuracy','status']])

## 7 — Plot measured single-card throughput and memory
Use only successful observed probes; do not treat estimated full-epoch seconds as measured.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
Path('figures').mkdir(exist_ok=True)
a=pd.read_csv('results/full_batch_scaling.csv');a=a[a.status=='ok']
fig,ax=plt.subplots();ax.plot(a.batch,a.samples_per_second,'o-');ax.set(xlabel='Batch size',ylabel='Samples / sec',title='Measured full-epoch batch throughput');fig.savefig('figures/throughput_vs_batch.png',bbox_inches='tight');plt.show()
b=pd.read_csv('results/full_memory_comparison.csv');b=b[(b.status=='ok') & (b.batch==int(c['batch_size']))].drop_duplicates('technique',keep='last')
fig,ax=plt.subplots(figsize=(9,4));ax.bar(b.technique,b.peak_gb);ax.tick_params(axis='x',rotation=30);ax.set(ylabel='Peak allocated GB',title='Measured full-epoch technique memory');fig.tight_layout();fig.savefig('figures/memory_comparison.png');plt.show()


## 8 — Part C: skipped (optional)

Per the instructor clarification, Part C is optional and is skipped. `METRICS.md` contains the exact phrase **Part C skipped (optional)**; the clarification text is kept in `INSTRUCTOR_CLARIFICATION.md`.

In [ ]:
print('Part C skipped (optional)')

## 9 — Part D: vLLM context / KV-cache sweep at batch 8 (one GPU)

Runs with `VLLM_PY`. A **fresh vLLM engine** is started at increasing `max_model_len` (512 → model limit) with batch 8; each run's full startup log is kept in `results/kv_logs/` (GPU-block count and block size are parsed when the installed vLLM prints them). Analytical KV bytes/token/sequence = `2 × layers × kv_heads × head_dim × dtype_bytes`. `nvidia-smi` shows vLLM's *reserved* memory, not KV occupancy — do not report it as KV usage. If the sweep reaches the model's context limit without an engine failure, report that, not "OOM"; the longest servable context at batch 8 then comes from `gpu_blocks × block_size ÷ 8`.

In [ ]:
r = subprocess.run([VLLM_PY,'src/kv_context_sweep.py'], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "kv_context_sweep.py failed — read the output above and results/kv_logs/*.log"
kv = pd.read_csv('results/kv_context_sweep.csv'); display(kv)
print((ROOT/'results/kv_context_summary.json').read_text())
assert (kv.status=='ok').any(), "No successful context length — the first engine start failed; inspect results/kv_logs/context_512.log"


## 10 — Part E: summary, RunPod evidence, and GPU-hours

Generate the measured GPU-hours log and preserve `nvidia-smi -L` output. Fill in qualitative written answers and record the instructor's optional-Part-C clarification. Use actual GPU identity, not a hard-coded RTX 4090 label.


In [ ]:
subprocess.run([sys.executable,'src/finalize_evidence.py'],check=True)
subprocess.run([sys.executable,'src/summarize_results.py'],check=False)
print((ROOT/'gpu_hours.csv').read_text())
print((ROOT/'GPU_INFO.txt').read_text()[:1500])


## 11 — Final checklist

- [ ] SEED = 2738 and SLICE identical to HW4.a; `data/twitter.csv` identical to HW4.a; exact row range in METRICS.md
- [ ] `old_seed42_run/` kept but **not** used in any table
- [ ] `GPU_INFO.txt` has real `nvidia-smi -L` output
- [ ] HW4.a curves, best epoch (comment on validation **loss** as well as accuracy), confusion matrix, five worst-class diagnoses rewritten for the **new** test set
- [ ] Table HW4.1 (HF b1, vLLM b1/8/32) + 3–4-sentence latency vs throughput answer
- [ ] Part A table + plot, plateau batch stated, A.3 marked N/A (one GPU, instructor waiver)
- [ ] Part B table with all **five** rows + best combination, ranked table by memory saved per throughput lost, accumulation equal-step comparison
- [ ] "Part C skipped (optional)" in METRICS.md; `INSTRUCTOR_CLARIFICATION.md` added
- [ ] Part D table, KV-vs-context result, longest context at batch 8 with stated reason (OOM vs model limit), comparison with earlier serving numbers
- [ ] Part E GPU-hours shown, chosen configuration justified, HW4.5 summary in METRICS.md (not an HW5 baseline)
- [ ] RUN_LOG.txt with pod start/stop and failures, AI_USE.md rewritten, gpu_hours.csv, git tag `hw4-5`

## 12 — Submission audit and theory notes
Run the audit script after experiments. It reports missing outputs; it does not fabricate results. Read `SUBMISSION_AUDIT.md` and `THEORY_VIVA_GUIDE.md`.

In [ ]:
subprocess.run([sys.executable,'src/submission_audit.py'],check=False)
print((ROOT/'results/audit_report.txt').read_text())
